# Review one tunnel image from Google Drive

**Experimental: live Colab interaction validation is pending.** Run cells individually; do not use Run all. Use a CPU runtime. Editing happens inside this notebook using Stackview and Matplotlib widgets.

Workflow: choose image → inspect the widget demo → confirm boundaries → automatic analysis → edit native crops → save reviewed result. The comparison runs separately. Authorize Drive only through the normal Colab prompt.

Your source TIFF is preserved. Drafts contain resumable checkpoints; final results contain only a CSV, overlay and audit ZIP. Apply saves edits; strokes not yet applied are lost on disconnect. One notebook should write a draft at a time.

In [ ]:
# 1. Settings — select code before importing the package.
SOURCE_FOLDER = "/content/drive/MyDrive/tunnel-quant/priority"
RESULTS_PARENT = "/content/drive/MyDrive/tunnel-quant/Reviewed"
RESUME_DRAFT = ""  # Paste the printed draft directory to reopen applied edits.
CODE_REVISION = "feature/notebook-mask-editor"  # branch, tag or exact commit
RUN_SENSITIVITY = False
REVIEWER = ""

In [ ]:
# 2. Connect Drive and install into the notebook kernel (widgets need this kernel).
import sys, subprocess, tempfile, json, os
from pathlib import Path
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or newer is required")
if "tunnel_analysis" in sys.modules:
    raise RuntimeError("Setup already imported this package. Continue below, or restart the session to change code.")
from google.colab import drive, output
drive.mount("/content/drive")
output.enable_custom_widget_manager()

saved_environment = None
if RESUME_DRAFT.strip():
    commits = sorted((Path(RESUME_DRAFT)/"commits").glob("*.json"))
    if not commits:
        raise ValueError("No completed draft found at RESUME_DRAFT")
    saved = json.loads(commits[-1].read_text())
    saved_environment = saved["environment"]
    CODE_REVISION = saved_environment["git_commit"]
    if not CODE_REVISION:
        raise ValueError("Draft has no Git commit; restore its original checkout manually")
    if saved_environment["python"] != __import__("platform").python_version():
        raise ValueError("This draft used a different Python version. Use its original environment to resume.")
    RUN_SENSITIVITY = bool(saved["sensitivity"])

if not CODE_REVISION or CODE_REVISION.startswith("-") or any(c.isspace() for c in CODE_REVISION):
    raise ValueError("Supply a branch, tag, or exact Git commit")
code = Path(tempfile.mkdtemp(prefix="tunnel-editor-code-",dir="/content"))
subprocess.run(["git","init",str(code)],check=True)
subprocess.run(["git","-C",str(code),"remote","add","origin","https://github.com/prateekdwv/tunnel-analysis.git"],check=True)
subprocess.run(["git","-C",str(code),"fetch","--depth","1","origin",CODE_REVISION],check=True)
subprocess.run(["git","-C",str(code),"checkout","--detach","FETCH_HEAD"],check=True)
commit = subprocess.check_output(["git","-C",str(code),"rev-parse","HEAD"],text=True).strip()
import importlib.metadata
modules = {"numpy":"numpy", "scipy":"scipy", "scikit-image":"skimage", "matplotlib":"matplotlib", "ipywidgets":"ipywidgets", "ipympl":"ipympl", "pillow":"PIL"}
loaded_versions = {name:importlib.metadata.version(name) for name,module in modules.items() if module in sys.modules}
packages = [f"{name}=={version}" for name,version in saved_environment["versions"].items()] if saved_environment else []
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(code)+"[notebook]",*packages],check=True)
changed = [name for name,version in loaded_versions.items() if importlib.metadata.version(name) != version]
if changed:
    raise RuntimeError("Restart the runtime and rerun setup: installed versions changed for already loaded packages: " + ", ".join(changed))
# The editable-install .pth takes effect only at interpreter startup; add it now.
sys.path.insert(0,str(code/"src"))
os.environ["MPLCONFIGDIR"] = "/content/tunnel-editor-mpl"
get_ipython().run_line_magic("matplotlib","widget")
print("Pinned code:",commit)
print("If Colab asks for a restart after installation, restart before continuing.")

In [ ]:
# 3. Choose exactly one TIFF. For resume the saved source is used.
import ipywidgets as widgets
from IPython.display import display
files = sorted((p for p in Path(SOURCE_FOLDER).iterdir() if p.is_file() and p.suffix.lower() in (".tif",".tiff")),key=lambda p:p.name)
if not files:
    raise ValueError("No TIFFs in SOURCE_FOLDER")
image_picker = widgets.Dropdown(options=[(p.name,str(p)) for p in files],description="Image:",layout=widgets.Layout(width="90%"))
display(image_picker)

In [ ]:
# 4. Open image or resume. Keep the printed draft path.
from tunnel_analysis.editor import EditSession, stamp
from tunnel_analysis.config import Config
if "session" in globals():
    if session.active_crop and session.active_crop.dirty:
        raise ValueError("Apply or discard unsaved edits before opening another image")
    for name in ("editor", "boundaries", "demo_brush"):
        if name in globals():
            globals()[name].close()
    session.close()
source = saved["source"] if RESUME_DRAFT.strip() else image_picker.value
draft = Path(RESUME_DRAFT) if RESUME_DRAFT.strip() else Path(RESULTS_PARENT)/".drafts"/("draft_"+stamp())
config = Config(**{**saved["config"],"ridge_sigmas":tuple(saved["config"]["ridge_sigmas"])}) if RESUME_DRAFT.strip() else Config()
session = EditSession(source,draft,scratch="/content",config=config,resume=bool(RESUME_DRAFT.strip()))
print("Image:",source)
print("Draft:",draft)
print("\n".join(session.notices))

## First-run widget check — before spending time on segmentation
Run the next cell. On the real image crop, test paint, erase, radius **1** and undo. Strokes should follow the pointer without lag or shifted pixels. This demo never saves changes. If the canvas is blank or edits are misplaced, stop and report the issue; do not proceed with measurements.

Then inspect the boundary controls in the following cell. Check dragging, resizing, rotation and magnified edges. Live browser responsiveness cannot be verified by the automated local tests.

In [ ]:
from tunnel_analysis.notebook_editor import brush_demo
if "demo_brush" in globals():
    demo_brush.close()
demo_widget, demo_brush = brush_demo(session.image)
display(demo_widget)

In [ ]:
# 5. Adjust the full arena, outer cutoff and optional inner exclusion.
from tunnel_analysis.notebook_editor import BoundaryEditor
if "boundaries" in globals():
    boundaries.close()
boundaries = BoundaryEditor(session)
display(boundaries.widget)

**Boundary controls:** Choose the ellipse from the dropdown. Drag handles to resize or move; numeric centre, radii and angle fields use native pixels and degrees. Turn on the circle constraint for equal radii. Enable the inner exclusion only when appropriate. Ellipses can have different centres.

Use the toolbar to pan towards an edge, then **Inspect selected edge** for a native-resolution view; **Whole image** restores the overview. White is the full coverage arena; blue outlines mark exclusions. Click **Confirm boundaries** before continuing. Changing boundaries invalidates the automatic result; previous applied work stays in the draft history.

In [ ]:
# 6. Automatic analysis. Prints progress; unchanged saved work is reused.
session.analyse(sensitivity=RUN_SENSITIVITY,progress=lambda line:print(line,flush=True))

In [ ]:
# 7. Edit crops and save each image independently.
from tunnel_analysis.notebook_editor import MaskEditor
if "editor" in globals():
    if session.active_crop and session.active_crop.dirty:
        raise ValueError("Apply or discard strokes before reopening the editor")
    editor.close()
editor = MaskEditor(session,RESULTS_PARENT,REVIEWER)
display(editor.widget)

**Editing:** Select a crop origin in the overview or enter its x/y coordinates; select crop size and display zoom, then **Open crop**. Keep crop size × zoom ≤1024 (256 at 4×, 512 at 2×, or 1024 at 1×). Paint adds pixels; Erase removes them. Radius 1 at a pixel centre edits exactly one pixel. Green counts, blue is excluded. Hide the mask to inspect the white tunnel edge. Dark gaps remain untouched unless you explicitly paint them.

**Undo stroke** affects the active crop. **Apply to Drive** commits its changes and verifies the checkpoint; **Discard strokes** restores the last applied crop. Apply or discard before switching crops. On disconnection, reopen the printed draft path using RESUME_DRAFT; applied crops survive without rerunning analysis. Keep the draft folder.

Once the whole image is reviewed, click **Save reviewed result**. Its new folder contains `results.csv`, `overlay.png`, and `audit.zip`. The audit keeps the automatic mask, corrected mask, boundaries and provenance. Review improves the mask but does not establish independent biological accuracy.

To work on another image, leave RESUME_DRAFT empty, choose another image in cell 3, and repeat from cell 4. Avoid reopening old widget outputs after changing the active session.

In [ ]:
# 8. Optional separate comparison: choose one reviewed result directory per image.
REVIEWED_RUNS = [
    # "/content/drive/MyDrive/tunnel-quant/Reviewed/reviewed_...",
]
if REVIEWED_RUNS:
    from tunnel_analysis.editor import aggregate_reviewed
    comparison = Path(RESULTS_PARENT)/("comparison_"+stamp())
    aggregate_reviewed(REVIEWED_RUNS,comparison)
    print("Comparison:",comparison)
    from IPython.display import Image
    display(Image(filename=str(comparison/"comparison.png")))
else:
    print("Add explicitly selected reviewed result folders when ready to compare.")